In [0]:

from pyspark.sql.functions import col, to_timestamp, to_date, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "workspace"
TARGET_SCHEMA = "insurance_lakehouse"

BRONZE_POLICY_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.bronze_policy"
BRONZE_CLAIMS_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.bronze_claims"
SILVER_POLICY_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.silver_policy"
SILVER_CLAIMS_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.silver_claims"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {TARGET_SCHEMA}")

In [0]:

def cast_if_exists(df, col_name, cast_type):
    if col_name in df.columns:
        return df.withColumn(col_name, col(col_name).cast(cast_type))
    return df

def to_timestamp_if_exists(df, col_name):
    if col_name in df.columns:
        return df.withColumn(col_name, to_timestamp(col(col_name)))
    return df

def to_date_if_exists(df, col_name):
    if col_name in df.columns:
        return df.withColumn(col_name, to_date(col(col_name)))
    return df

def latest_per_key(df, key_col, order_col="updated_ts"):
    if key_col not in df.columns:
        raise ValueError(f"Expected key column '{key_col}' not found. Found columns: {df.columns}")

    order_exprs = []
    if order_col in df.columns:
        order_exprs.append(col(order_col).desc_nulls_last())
    if "ingestion_ts" in df.columns:
        order_exprs.append(col("ingestion_ts").desc_nulls_last())

    if not order_exprs:
        raise ValueError(f"Need at least one ordering column like '{order_col}' or 'ingestion_ts'")

    w = Window.partitionBy(key_col).orderBy(*order_exprs)

    return (
        df.withColumn("rn", row_number().over(w))
          .filter(col("rn") == 1)
          .drop("rn")
    )

def upsert_delta(source_df, target_table, business_key):
    if not spark.catalog.tableExists(target_table):
        (
            source_df.write.format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )
        print(f"Created {target_table}")
        return

    target = DeltaTable.forName(spark, target_table)

    (
        target.alias("t")
        .merge(source_df.alias("s"), f"t.{business_key} = s.{business_key}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Merged into {target_table}")

In [0]:

# ----------------------------
# Policy Silver
# Assumes your generated CSV has policy_id and updated_ts
# ----------------------------
policy_df = spark.table(BRONZE_POLICY_TABLE)

policy_df = to_timestamp_if_exists(policy_df, "updated_ts")
policy_df = to_date_if_exists(policy_df, "policy_start_date")
policy_df = to_date_if_exists(policy_df, "policy_end_date")
policy_df = to_date_if_exists(policy_df, "snapshot_date")

policy_latest_df = latest_per_key(policy_df, key_col="policy_id", order_col="updated_ts")

upsert_delta(policy_latest_df, SILVER_POLICY_TABLE, "policy_id")

display(spark.table(SILVER_POLICY_TABLE))

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4771439838087021>, line 6
      1 # Databricks notebook source
      2 # ----------------------------
      3 # Policy Silver
      4 # Assumes your generated CSV has policy_id and updated_ts
      5 # ----------------------------
----> 6 policy_df = spark.table(BRONZE_POLICY_TABLE)
      8 policy_df = to_timestamp_if_exists(policy_df, "updated_ts")
      9 policy_df = to_date_if_exists(policy_df, "policy_start_date")

NameError: name 'BRONZE_POLICY_TABLE' is not defined

In [0]:

# ----------------------------
# Claims Silver
# Assumes your generated CSV has claim_id and updated_ts
# ----------------------------
claims_df = spark.table(BRONZE_CLAIMS_TABLE)

claims_df = to_timestamp_if_exists(claims_df, "updated_ts")
claims_df = to_date_if_exists(claims_df, "claim_date")
claims_df = to_date_if_exists(claims_df, "snapshot_date")

claims_latest_df = latest_per_key(claims_df, key_col="claim_id", order_col="updated_ts")

upsert_delta(claims_latest_df, SILVER_CLAIMS_TABLE, "claim_id")

display(spark.table(SILVER_CLAIMS_TABLE))

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4771439838087022>, line 6
      1 # Databricks notebook source
      2 # ----------------------------
      3 # Claims Silver
      4 # Assumes your generated CSV has claim_id and updated_ts
      5 # ----------------------------
----> 6 claims_df = spark.table(BRONZE_CLAIMS_TABLE)
      8 claims_df = to_timestamp_if_exists(claims_df, "updated_ts")
      9 claims_df = to_date_if_exists(claims_df, "claim_date")

NameError: name 'BRONZE_CLAIMS_TABLE' is not defined

In [0]:
display(spark.table("workspace.insurance_lakehouse.silver_policy"))
display(spark.table("workspace.insurance_lakehouse.silver_claims"))